# 15주차 70% 프로젝트 프로토타입 미션

14주차의 작동 경로를 보존한 뒤 서로 다른 두 수정 행동을 적용하고, 수정 전후 결과와 재실행 가능한 증거를 남깁니다. 기본 `provided` 경로는 외부 파일 없이 실행됩니다. 자신의 프로젝트를 이어갈 때에는 교수에게 승인받은 입력과 코드만 `APPROVED PROJECT CODE ZONE`에 연결합니다.

필수 제출 파일은 실행 결과가 남은 노트북, 개선 결과 PNG, 수정 기록 HTML입니다. 자신의 입력 파일이 필요하면 같은 이름으로 함께 제출합니다.

In [ ]:
# STEP 0 · 실행 환경과 공통 도구
from pathlib import Path
from hashlib import sha256
from html import escape
import base64
import re
import shutil

import matplotlib.pyplot as plt
from PIL import Image

_week15_step0_runs = globals().get("_week15_step0_runs", 0) + 1
_run_order = [0]

def safe_fragment(value):
    cleaned = re.sub(r"[^0-9A-Za-z가-힣_-]+", "_", str(value).strip())
    return cleaned.strip("_")

def sha256_file(path):
    return sha256(Path(path).read_bytes()).hexdigest()

def ensure_written(value, label, minimum=8):
    text = str(value).strip()
    assert text and "EDIT:" not in text, f"{label}을 자신의 문장으로 작성하세요."
    assert len(text) >= minimum, f"{label}을 더 구체적으로 작성하세요."
    return text

print("STEP 0 READY")

In [ ]:
# STEP 1 · EDIT: 제출 정보, 프로젝트 계약, 자료 책임
student_id = "학번"
student_name = "이름"
project_track = "data"  # data / text / sound / image
project_mode = "provided"  # provided / own
own_source_filename = ""  # own일 때 승인된 실제 입력 파일명
baseline_mode = "provided"  # provided / upload
baseline_source_filename = ""  # upload일 때 14주차 PNG 또는 HTML

approval_status = "EDIT: provided 또는 approved"
project_question = "EDIT: 결과가 답할 수 있는 질문 한 문장"
intended_audience = "EDIT: 이 결과를 읽을 구체적인 대상"
source_title = "EDIT: 입력 자료의 제목과 제공자"
usage_rights = "EDIT: 수업 제출에 사용할 수 있는 근거"
reference_date = "EDIT: 자료의 기준일 또는 제작일"
privacy_check = "EDIT: 개인정보 포함 여부와 제외 기준"

assert project_track in {"data", "text", "sound", "image"}
assert project_mode in {"provided", "own"}
assert baseline_mode in {"provided", "upload"}
assert approval_status in {"provided", "approved"}, "approval_status는 provided 또는 approved여야 합니다."
expected_approval = "provided" if project_mode == "provided" else "approved"
assert approval_status == expected_approval, "provided 경로는 provided, own 경로는 approved 판정이 필요합니다."
_run_order.append(1)
print("STEP 1 RECORDED")

In [ ]:
# STEP 2 · 14주차 기준점과 15주차 파일명 준비
safe_student_id = safe_fragment(student_id)
safe_student_name = safe_fragment(student_name)
assert safe_student_id not in {"", "학번"}, "실제 학번을 입력하세요."
assert safe_student_name not in {"", "이름"}, "실제 이름을 입력하세요."

baseline_output_path = Path(
    f"week15_{safe_student_id}_{safe_student_name}_baseline.png"
)
refined_output_path = Path(
    f"week15_{safe_student_id}_{safe_student_name}_refined.png"
)
revision_log_path = Path(
    f"week15_{safe_student_id}_{safe_student_name}_revision_log.html"
)

own_source_path = None
own_source_digest = None
if project_mode == "own":
    own_source_path = Path(own_source_filename)
    assert own_source_path.is_file(), "own 경로의 승인된 실제 입력 파일을 찾을 수 없습니다."
    own_source_digest = sha256_file(own_source_path)

if baseline_mode == "upload":
    baseline_source_path = Path(baseline_source_filename)
    assert baseline_source_path.is_file(), "14주차 기준점 파일을 찾을 수 없습니다."
    baseline_suffix = baseline_source_path.suffix.lower()
    assert baseline_suffix in {".png", ".html"}, "기준점은 14주차 PNG 또는 HTML이어야 합니다."
    if baseline_suffix == ".png":
        shutil.copyfile(baseline_source_path, baseline_output_path)
    else:
        baseline_html = baseline_source_path.read_text(encoding="utf-8")
        encoded_match = re.search(
            r"data:image/png;base64,([A-Za-z0-9+/=]+)", baseline_html
        )
        assert encoded_match, "14주차 HTML에서 포함된 PNG 기준점을 찾을 수 없습니다."
        baseline_png_bytes = base64.b64decode(
            encoded_match.group(1), validate=True
        )
        assert baseline_png_bytes.startswith(b"\x89PNG\r\n\x1a\n"), "HTML 기준점의 포함 이미지는 PNG여야 합니다."
        baseline_output_path.write_bytes(baseline_png_bytes)

_run_order.append(2)
print("STEP 2 PATHS READY")

In [ ]:
# STEP 3 · EDIT: 두 수정 행동과 승인된 프로젝트 코드
revision_focus_1 = "accuracy"
revision_focus_2 = "readability"
revision_action_1 = "EDIT: 정확성: 값과 화면의 일치를 높이는 수정"
revision_action_2 = "EDIT: 가독성: 제목, 단위 또는 설명을 높이는 수정"

assert {revision_focus_1, revision_focus_2} == {"accuracy", "readability"}
ensure_written(revision_action_1, "수정 행동 1")
ensure_written(revision_action_2, "수정 행동 2")
assert revision_action_1.startswith("정확성:"), "수정 행동 1은 '정확성:'으로 시작하세요."
assert revision_action_2.startswith("가독성:"), "수정 행동 2는 '가독성:'으로 시작하세요."
revision_action_detail_1 = revision_action_1.split(":", 1)[1].strip()
revision_action_detail_2 = revision_action_2.split(":", 1)[1].strip()
ensure_written(revision_action_detail_1, "수정 행동 1의 구체적 내용")
ensure_written(revision_action_detail_2, "수정 행동 2의 구체적 내용")
assert revision_action_detail_1 != revision_action_detail_2, "서로 다른 두 수정 행동을 기록하세요."
revision_evidence_id = sha256(
    f"{revision_action_1}\n{revision_action_2}".encode("utf-8")
).hexdigest()[:12]

# APPROVED PROJECT CODE ZONE
# 자신의 프로젝트를 이어갈 때에는 이 함수 안의 제공 예시만 승인된 코드로 교체합니다.
# own 경로는 input_origin="own", own_source_digest와 실제 적용한 두 revision focus를 증거로 반환해야 합니다.
# 함수는 수정 전 Figure, 수정 후 Figure, 화면 증거 사전을 반환해야 합니다.
def build_project_outputs():
    baseline_figure, baseline_axis = plt.subplots(figsize=(8, 5), dpi=200)
    refined_figure, refined_axis = plt.subplots(figsize=(8, 5), dpi=200)

    if project_track == "data":
        labels = ["Archive", "Studio", "Screening"]
        raw_values = [84, 63, 49]
        refined_values = list(raw_values)
        baseline_axis.barh(labels, raw_values, color="#b8b9b2")
        bars = refined_axis.barh(
            labels[::-1],
            refined_values[::-1],
            color=["#116e68", "#6f4f00", "#a23d34"][::-1],
        )
        refined_axis.bar_label(
            bars,
            labels=[str(value) for value in refined_values[::-1]],
            padding=5,
        )
        refined_axis.set_xlim(0, 95)
        refined_axis.set_xlabel("Total visits")
        unit = "Total visits"
    elif project_track == "text":
        labels = ["room", "record", "trace", "light"]
        raw_values = [3, 8, 4, 6]
        baseline_axis.bar(labels, raw_values, color="#b8b9b2")
        ranked = sorted(zip(labels, raw_values), key=lambda item: item[1])
        refined_labels = [label for label, _value in ranked]
        refined_values = [value for _label, value in ranked]
        bars = refined_axis.barh(
            refined_labels, refined_values, color="#365f91"
        )
        refined_axis.bar_label(bars, padding=5)
        refined_axis.set_xlim(0, 9)
        refined_axis.set_xlabel("Occurrences")
        unit = "Occurrences"
    elif project_track == "sound":
        labels = ["0.0", "0.5", "1.0", "1.5", "2.0", "2.5", "3.0", "3.5"]
        raw_values = [0.08, 0.16, 0.58, 0.22, 0.13, 0.31, 0.72, 0.18]
        refined_values = list(raw_values)
        time_values = [index * 0.5 for index in range(len(raw_values))]
        baseline_axis.plot(time_values, raw_values, color="#b8b9b2")
        refined_axis.plot(
            time_values,
            refined_values,
            color="#116e68",
            marker="o",
            linewidth=2,
        )
        peak_index = max(
            range(len(refined_values)), key=refined_values.__getitem__
        )
        refined_axis.annotate(
            f"peak {time_values[peak_index]:.1f}s",
            (time_values[peak_index], refined_values[peak_index]),
            xytext=(-54, 20),
            textcoords="offset points",
            arrowprops={"arrowstyle": "->", "color": "#202523"},
        )
        refined_axis.set_xlabel("Time (seconds)")
        refined_axis.set_ylabel("Relative RMS")
        refined_axis.set_ylim(0, 0.82)
        unit = "Relative RMS"
    else:
        labels = ["A", "B", "C", "D", "E"]
        raw_values = [52, 76, 44, 66, 58]
        refined_values = list(raw_values)
        x_values = [0.16, 0.34, 0.52, 0.70, 0.84]
        y_values = [0.26, 0.68, 0.43, 0.72, 0.31]
        colors = ["#116e68", "#365f91", "#a23d34", "#6f4f00", "#116e68"]
        baseline_axis.scatter(
            x_values, y_values, s=[value * 10 for value in raw_values], c=colors
        )
        refined_axis.scatter(
            x_values,
            y_values,
            s=[value * 10 for value in refined_values],
            c=colors,
            edgecolors="#202523",
            linewidths=1.5,
        )
        for label, x_value, y_value in zip(labels, x_values, y_values):
            refined_axis.text(x_value, y_value, label, ha="center", va="center", color="white", fontweight="bold")
        refined_axis.set_xlim(0, 1)
        refined_axis.set_ylim(0, 1)
        refined_axis.set_aspect("equal")
        refined_axis.set_xlabel("Normalized x position")
        refined_axis.set_ylabel("Normalized y position")
        unit = "Shape size"

    baseline_axis.set_title(f"{project_track.upper()} · BASELINE")
    refined_axis.set_title(f"{project_track.upper()} · REFINED")
    refined_axis.spines[["top", "right"]].set_visible(False)
    refined_figure.text(
        0.01,
        0.035,
        f"APPLIED: {revision_focus_1.upper()} + {revision_focus_2.upper()} · {revision_evidence_id}",
        fontsize=7,
    )
    refined_figure.text(0.01, 0.012, f"Source: {source_title} | Date: {reference_date}", fontsize=7)
    refined_figure.tight_layout(rect=(0, 0.065, 1, 1))

    evidence = {
        "track": project_track,
        "input_origin": "provided",
        "input_digest": None,
        "revision_evidence_id": revision_evidence_id,
        "applied_revision_focuses": [
            revision_focus_1,
            revision_focus_2,
        ],
        "input_count": len(raw_values),
        "visual_count": len(refined_values),
        "value_match": sorted(raw_values) == sorted(refined_values),
        "unit": unit,
    }
    return baseline_figure, refined_figure, evidence

baseline_figure, refined_figure, project_evidence = build_project_outputs()
if baseline_mode == "provided":
    baseline_figure.savefig(baseline_output_path, dpi=200, facecolor="#f3efe5")
plt.close(baseline_figure)
refined_figure.savefig(refined_output_path, dpi=200, facecolor="#f3efe5")
plt.close(refined_figure)

baseline_snapshot_digest = sha256_file(baseline_output_path)
refined_output_digest = sha256_file(refined_output_path)
_run_order.append(3)
print("STEP 3 OUTPUTS SAVED")

In [ ]:
# STEP 4 · 자동 증거와 수정 기록 HTML 만들기
with Image.open(baseline_output_path) as baseline_image:
    assert baseline_image.size == (1600, 1000), "기준점 PNG는 1600 x 1000이어야 합니다."
    baseline_format = baseline_image.format
with Image.open(refined_output_path) as saved_image:
    assert saved_image.size == (1600, 1000), "개선 PNG는 1600 x 1000이어야 합니다."
    refined_format = saved_image.format

assert baseline_format == "PNG" and refined_format == "PNG"
assert baseline_snapshot_digest != refined_output_digest, "수정 전후 파일이 같습니다. 두 수정 행동을 실제 화면에 반영하세요."
required_evidence_fields = {
    "track",
    "input_origin",
    "input_digest",
    "revision_evidence_id",
    "applied_revision_focuses",
    "input_count",
    "visual_count",
    "value_match",
    "unit",
}
assert required_evidence_fields <= project_evidence.keys(), "승인 코드의 화면 증거 필드가 부족합니다."
assert project_evidence["track"] == project_track
assert project_evidence["input_origin"] == project_mode, "own 경로는 승인 코드가 input_origin='own' 증거를 반환해야 합니다."
assert project_evidence["input_digest"] == own_source_digest, "own 경로는 실제 입력 파일의 SHA-256 증거를 반환해야 합니다."
assert project_evidence["revision_evidence_id"] == revision_evidence_id, "수정 행동 문장과 결과의 증거 ID가 일치해야 합니다."
assert set(project_evidence["applied_revision_focuses"]) == {"accuracy", "readability"}, "정확성과 가독성 수정이 모두 실제 결과에 적용되어야 합니다."
assert project_evidence["input_count"] == project_evidence["visual_count"]
assert project_evidence["value_match"] is True

evidence_report = {
    "baseline_digest": baseline_snapshot_digest,
    "refined_digest": refined_output_digest,
    "baseline_size": (1600, 1000),
    "refined_size": (1600, 1000),
    "input_count": project_evidence["input_count"],
    "visual_count": project_evidence["visual_count"],
    "value_match": project_evidence["value_match"],
    "input_origin": project_evidence["input_origin"],
    "input_digest": project_evidence["input_digest"],
    "revision_evidence_id": project_evidence[
        "revision_evidence_id"
    ],
    "applied_revision_focuses": project_evidence[
        "applied_revision_focuses"
    ],
}

_run_order.append(4)
print("AUTOMATIC EVIDENCE READY · TEACHER CHECK REQUIRED")

In [ ]:
# STEP 5 · EDIT: 관찰, 한계, 교수 피드백과 확인 상태
main_observation = "EDIT: 수정 후 화면에서 직접 가리킬 수 있는 관찰"
limitation_statement = "EDIT: 현재 입력과 표현만으로 단정할 수 없는 한계"
teacher_feedback = "EDIT: 교수 확인에서 합의한 유지 또는 수정 내용"
teacher_gate = "pending"  # 확인 뒤 confirmed

baseline_data_uri = "data:image/png;base64," + base64.b64encode(
    baseline_output_path.read_bytes()
).decode("ascii")
refined_data_uri = "data:image/png;base64," + base64.b64encode(
    refined_output_path.read_bytes()
).decode("ascii")

revision_log_html = f'''<!doctype html>
<html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>{escape(project_question)}</title><style>
body{{max-width:1080px;margin:0 auto;padding:32px;font:16px/1.65 system-ui,sans-serif;color:#202523;background:#f3efe5}}
main{{padding:28px;border:1px solid #c7c8be;background:#fffdf8}}img{{width:100%;height:auto;border:1px solid #c7c8be}}
.compare{{display:grid;grid-template-columns:1fr 1fr;gap:18px}}dt{{font-weight:700}}dd{{margin:0 0 12px}}@media(max-width:760px){{.compare{{grid-template-columns:1fr}}}}
</style></head><body><main><h1>{escape(project_question)}</h1>
<div class="compare"><figure><img src="{baseline_data_uri}" alt="수정 전 결과"><figcaption>수정 전</figcaption></figure>
<figure><img src="{refined_data_uri}" alt="수정 후 결과"><figcaption>수정 후</figcaption></figure></div>
<dl><dt>수정 행동 1</dt><dd>{escape(revision_action_1)}</dd><dt>수정 행동 2</dt><dd>{escape(revision_action_2)}</dd>
<dt>수정 증거 ID</dt><dd>{revision_evidence_id}</dd><dt>관찰</dt><dd>{escape(main_observation)}</dd><dt>한계</dt><dd>{escape(limitation_statement)}</dd>
<dt>출처</dt><dd>{escape(source_title)} / {escape(usage_rights)} / {escape(reference_date)}</dd>
<dt>개인정보 점검</dt><dd>{escape(privacy_check)}</dd><dt>교수 피드백</dt><dd>{escape(teacher_feedback)}</dd></dl></main></body></html>'''
revision_log_path.write_text(revision_log_html, encoding="utf-8")

_run_order.append(5)
print("STEP 5 INTERPRETATION RECORDED")

In [ ]:
# STEP 6 · FINAL CHECK: 수정하지 않습니다
_run_order.append(6)
assert _week15_step0_runs == 1, "마지막 검사는 새 런타임에서 모두 실행해야 합니다."
assert _run_order == [0, 1, 2, 3, 4, 5, 6], "새 런타임에서 위에서 아래로 한 번씩 실행하세요."

for value, label in [
    (approval_status, "승인 상태"),
    (project_question, "프로젝트 질문"),
    (intended_audience, "예상 독자"),
    (source_title, "자료 제목과 제공자"),
    (usage_rights, "이용 근거"),
    (reference_date, "기준일"),
    (privacy_check, "개인정보 점검"),
    (revision_action_1, "수정 행동 1"),
    (revision_action_2, "수정 행동 2"),
    (main_observation, "관찰"),
    (limitation_statement, "한계"),
    (teacher_feedback, "교수 피드백"),
]:
    ensure_written(value, label)

assert revision_action_1.startswith("정확성:"), "수정 행동 1은 '정확성:'으로 시작하세요."
assert revision_action_2.startswith("가독성:"), "수정 행동 2는 '가독성:'으로 시작하세요."
assert revision_action_detail_1 != revision_action_detail_2, "서로 다른 두 수정 행동을 기록하세요."
assert evidence_report["baseline_digest"] != evidence_report["refined_digest"]
assert evidence_report["value_match"] is True
assert evidence_report["input_origin"] == project_mode
assert evidence_report["input_digest"] == own_source_digest
assert evidence_report["revision_evidence_id"] == revision_evidence_id
if project_mode == "own":
    assert sha256_file(own_source_path) == own_source_digest, "실행 중 own 입력 파일이 변경되었습니다."
assert set(evidence_report["applied_revision_focuses"]) == {"accuracy", "readability"}
assert baseline_output_path.is_file()
assert refined_output_path.is_file()
assert revision_log_path.is_file()
assert teacher_gate == "confirmed", "교수의 증거 확인 뒤 teacher_gate를 confirmed로 바꾸세요."

print("PASS 16/16")
print("WEEK 15 PROJECT REFINEMENT COMPLETE")
print("제출:", refined_output_path.name, revision_log_path.name)